In [1]:
import os, sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [2]:
from pyspark.sql import SparkSession, functions as F, Window
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))   # notebook's folder -> its parent

from practice_data import TABLES, typed_rows, EXPECTED_COUNTS

spark = (SparkSession.builder
         .master("local[*]")
         .appName("sql-patterns")
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.python.worker.faulthandler.enabled", "true")
         .config("spark.sql.execution.pyspark.udf.faulthandler.enabled", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

def load_spark(name):
    ddl = ", ".join(f"{c} {k}" for c, k in TABLES[name]["types"].items())
    df = spark.createDataFrame(typed_rows(name), schema=ddl)
    df.createOrReplaceTempView(name)    # also lets you run spark.sql("SELECT ...")
    return df

customers = load_spark("customers_dedup")
scores    = load_spark("scores")
orders    = load_spark("rt_orders")
items     = load_spark("rt_order_items")

# sanity check
for name, df in [("customers_dedup", customers), ("scores", scores),
                 ("rt_orders", orders), ("rt_order_items", items)]:
    assert df.count() == EXPECTED_COUNTS[name], name
print("PySpark data loaded")

PySpark data loaded


In [3]:
customers.show(5)

+-----------+-----------------+---------+--------+----------+
|customer_id|            email|     name|   phone|updated_at|
+-----------+-----------------+---------+--------+----------+
|          3|  bob@example.com|Bob Brown|555-2002|2024-01-05|
|          5|  bob@example.com|    Bob B|555-2002|2024-01-25|
|          6|david@example.com|  David D|555-4004|2024-01-12|
|          8|  eve@example.com|    Eve E|555-5005|2024-01-08|
|         11|  bob@example.com| Bob Beta|555-2002|2024-01-28|
+-----------+-----------------+---------+--------+----------+
only showing top 5 rows
